# Week 1 · Day 1 — Website Summarizer

A first look at calling an LLM from Python. We'll:

1. Set up the OpenAI client and load our API key.
2. Send a simple chat completion ("tell me a joke").
3. Scrape a real web page down to clean text.
4. Ask the model to summarize that page in markdown.

> **Prerequisite:** copy `.env.example` to `.env` and add your `OPENAI_API_KEY`.

---

## 1. Setup & imports

In [ ]:
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from scraper import clean_html, fetch_website_content

## 2. Load API credentials

`load_dotenv` reads the key/value pairs from our local `.env` file into
environment variables. We fail fast if the OpenAI key is missing.

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set")

## 3. Your first chat completion

The Chat Completions API takes a list of `messages`. Each message has a
`role` (`system`, `user`, or `assistant`) and `content`. Here we send a
single user message and print the model's reply.

In [ ]:
message = "Hi GPT it's my first time using you, can you tell me a joke?"
messages = [
    {"role": "user", "content": message},
]

In [ ]:
# Instantiate the client. It automatically picks up OPENAI_API_KEY from the
# environment, so no key needs to be passed explicitly.
openai = OpenAI()

response = openai.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,  # type: ignore
)

# The reply text lives at choices[0].message.content.
display(Markdown(f"**GPT:** {response.choices[0].message.content}"))

## 4. Scrape a website

Before we can summarize a page, we need its content as plain text.
`fetch_website_content` (defined in [scraper.py](scraper.py)) downloads the
page, strips out scripts/styles/images, and returns the title plus visible
text.

In [ ]:
my_website_content = fetch_website_content(url="https://azam-sys.netlify.app/")
print(my_website_content)

## 5. Summarize the page with an LLM

This is a classic **system + user prompt** pattern:

- The **system prompt** sets the model's role and output format.
- The **user prompt** carries the actual data (the scraped page text).

In [ ]:
# Defines the assistant's role and the format we want back. No interpolation
# needed here, so it's a plain string (not an f-string).
system_prompt = (
    "You are a helpful assistant that summarizes website content. "
    "Provide a concise summary of the main points and topics covered on the "
    "website. Respond in markdown format."
)

In [ ]:
# The user prompt carries the scraped page text for the model to summarize.
user_prompt = f"Here is the content of the website: {my_website_content}"

In [ ]:
from typing import Dict, List


def summarize_website_content(messages: List[Dict[str, str]]) -> str | None:
    """Send a prepared messages list to the model and return the reply text.

    Args:
        messages: Chat messages (system + user) describing the task and data.

    Returns:
        The model's response content, or ``None`` if the model returned no text.
    """
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=messages,  # type: ignore
    )
    return response.choices[0].message.content

In [ ]:
# Combine both prompts into the messages list the API expects.
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [ ]:
# Send the system + user prompts together and render the markdown summary.
response = summarize_website_content(messages)

display(Markdown(data=f"**GPT:** {response}"))

## 6. Reuse the helper on another page

Now that `summarize_website_content` is a function, summarizing a different
site is just: scrape → build messages → call the helper.

In [ ]:
content = fetch_website_content(url="https://cnn.com/")

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Here is the content of the website: {content}"},
]

In [ ]:
summary = summarize_website_content(messages)
display(Markdown(data=f"**GPT:** {summary}"))

## 7. Selenium-based web scraping

The `requests`-based scraper only sees the initial HTML. For sites that
render their content with JavaScript (single-page apps, infinite scroll,
etc.), we need a real browser. **Selenium** drives a real Chrome instance so
we capture the fully-rendered page, then we reuse `clean_html` to keep the
token count down.

The `WebsiteSummarizer` class below bundles the whole flow — fetch, summarize,
display — behind a small object.

> **Requires:** `selenium` (already in `requirements.txt`). Selenium 4.6+ auto-downloads
> a matching chromedriver, so no manual driver setup is needed — just a local Chrome install.
> A browser window will open while it loads; add `options.add_argument("--headless=new")`
> to run without one.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options


class WebsiteSummarizer:
    """Fetch a JS-rendered page with Selenium and summarize it with an LLM.

    Typical usage:
        ws = WebsiteSummarizer(url)
        ws.selenium_fetch_content()
        ws.summarize_content()
        ws.display_summary()
    """

    def __init__(self, url: str, base_url: str = "https://api.openai.com/v1", api_key: str | None = None, model: str = "gpt-5-nano") -> None:
        self.url = url
        self.html: str | None = None 
        self.summary: str | None = None 
        self.client = OpenAI(base_url=base_url, api_key=api_key)
        self.model = model

    def selenium_fetch_content(self) -> None:
        """Load the page in headless Chrome and capture the cleaned text."""
        options = Options()
        options.add_argument(
            "User-Agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        )
        driver = webdriver.Chrome(options=options)
        try:
            driver.get(url=self.url)
            driver.implicitly_wait(5)
            # Strip the rendered HTML down to title + visible text so we send far fewer tokens to the model.
            self.html = clean_html(html=driver.page_source)
        except Exception as e:
            print(f"Error fetching content with Selenium: {e}")
        finally:
            driver.quit()

    def summarize_content(self) -> None:
        """Summarize the fetched content using the LLM."""
        if not self.html:
            raise ValueError(
                "Content not fetched yet. Call selenium_fetch_content() first."
            )

        system_prompt = (
            "You are a helpful assistant that summarizes website content. "
            "Provide a concise summary of the main points and topics covered on the "
            "website. Respond in markdown format."
        )
        user_prompt = f"Here is the content of the website: {self.html}"

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,  # type: ignore
        )
        self.summary = response.choices[0].message.content

    def display_summary(self) -> None:
        """Render the summary as markdown."""
        if not self.summary:
            raise ValueError(
                "Summary not generated yet. Call summarize_content() first."
            )
        display(Markdown(data=self.summary))

In [ ]:
ws = WebsiteSummarizer(url="https://www.netflix.com/")
ws.selenium_fetch_content()
ws.summarize_content()
ws.display_summary()

## 8. Point the summarizer at any OpenAI-compatible provider

`WebsiteSummarizer` now takes `base_url`, `api_key`, and `model`, so the exact
same scrape → summarize → display flow can run against a **local model via
Ollama** (or any OpenAI-compatible endpoint) instead of OpenAI — no code
changes, just different constructor arguments.

In [ ]:
ws = WebsiteSummarizer(url="https://dhan.co", base_url="http://localhost:11434/v1", api_key="ollama", model="llama3.2:3b")
ws.selenium_fetch_content()
ws.summarize_content()
ws.display_summary()